## DSAN 6000 Homework 3A: Allocating Tasks to Parallel Workers with `joblib`

## Overview

You made it to the first DSAN 6000 homework introducing a new coding concept! The goal of this part is for you to gain hands-on experience using **`joblib`** to quickly parallelize an **embarrassingly-parallel** task.

In this case, the problem is one that relates back to Week 1 and Week 2 content on the **OnLine Transaction Processing (OLTP)** mode of data collection and processing: the setup is that a designer spoon brand has set up a new website, where users from across the globe can purchase their latest avant garde silverware using their credit cards. Information from these credit card transactions then **streams into their OLTP database** like you saw in the Week 1 demo!

...However, you've been hired by this designer spoon brand (given your reputation as a DSAN graduate expert) because the company has been experiencing an upsurge in the use of **fraudulent (fake) credit card numbers** being used during checkout. so your task will be to use **parallel processing** to see how/whether you can "push" your OLTP system to process a large volume of transactions in a short amount of time.

Thus, this basic task was chosen specifically so that you can take your intuitions about how long it might take for a **serial** algorithm to carry out these conversions, and compare with how quickly you'll be able to complete it using `joblib` to **distribute** the subtasks to different **workers**, who can process the currencies and amounts in parallel.

## Part 1: Loading and Preparing Data

### Part 1.1: Use `boto3` to Download the `.parquet` File From S3

Since you already figured out how to use `boto3` to connect to and download files from an S3 bucket in HW2, here we have provided code for you importing `boto3` and setting up the `s3` client object. Your task is to use this client to **download** the file at the following URI:

```
s3://dsan6000-data/transactions_10m.parquet
```

To your EC2 instance, saving it to have the same filename in a `data` subfolder (within the `dsan6000-hw03-parallel-processing` folder). We have already included a `.gitignore` file in the template repo, to ensure that this `.parquet` file doesn't get pushed to GitHub, since it is larger than the allowed invidual file size on GitHub!

In [1]:
#| label: q1.1-init
import boto3
s3 = boto3.client('s3')

In [2]:
#| label: q1.1-response
# Your code here: Download transactions_10m.parquet to the data subfolder
s3.download_file('dsan6000-data', 'transactions_10m.parquet', 'data/transactions_10m.parquet')

### Part 1.2: Loading the Data Into Pandas

In [3]:
#| label: q1.2-init
import pandas as pd

In [4]:
#| label: q1.2-response
oltp_df = pd.read_parquet("data/transactions_10m.parquet")
oltp_df

,timestamp,customer_id,product_id,amount
0,2026-08-20 09:11:41.546837,13927,59,70.30 PLN
1,2026-08-20 09:11:48.636365,96584,45,47.90 JPY
2,2026-08-20 09:11:48.713547,85012,70,21.67 ZAR
3,2026-08-20 09:11:48.925565,96518,36,81.23 JPY
4,2026-08-20 09:11:55.104031,29571,29,76.49 DKK
...,...,...,...,...
9999995,2026-09-04 09:09:31.340697,70218,3,29.09 USD
9999996,2026-09-04 09:09:31.411286,62114,65,66.59 MXN
9999997,2026-09-04 09:09:31.554090,97054,29,32.54 JPY
9999998,2026-09-04 09:09:31.560754,55995,67,35.19 KRW


## Part 2: Converting Currencies in Serial

Here, the goal is explicitly *not* to do anything fancy: just use a standard for loop to process each of the 20 million transactions you just downloaded. To make the comparison with parallel processing as fair as possible, however, you should **extract just the `amount` column** from the full `DataFrame`, using the `to_list()` function available on Pandas `Series` objects to store these extracted values in a list named `amounts`.

In [5]:
import time
disp_time = lambda start, end: print('{:.4f} s'.format(end - start))

class MyConverter:
  def __init__(self):
    self.conversion_rates = {
      'USD': 0.865726, 'JPY': 0.005602, 'CZK': 0.041162, 'DKK': 0.133774,
      'GBP': 1.168252, 'HUF': 0.002737, 'PLN': 0.230319, 'RON': 0.190230,
      'SEK': 0.088645, 'CHF': 1.060333, 'ISK': 0.007153, 'NOK': 0.092876,
      'TRY': 0.017805, 'AUD': 0.617208, 'BRL': 0.167887, 'CAD': 0.623403,
      'CNY': 0.129051, 'HKD': 0.110376, 'IDR': 0.000049, 'INR': 0.009060,
      'KRW': 0.000643, 'MXN': 0.050710, 'MYR': 0.212395, 'NZD': 0.499700,
      'PHP': 0.013771, 'SGD': 0.681385, 'THB': 0.026037, 'ZAR': 0.053278,
    }

  def convert(self, amount: str):
    currency_elts = amount.split(" ")
    currency_amt = float(currency_elts[0])
    currency_code = currency_elts[1]
    return self.conversion_rates[currency_code] * currency_amt


In [6]:
converter = MyConverter()

In [7]:
converter.convert('93.60 USD')

81.0319536

In [8]:
amounts = oltp_df['amount'].to_list()

In [9]:
serial_start = time.time()
amounts_converted = [converter.convert(a) for a in amounts]
serial_end = time.time()
disp_time(serial_start, serial_end)

3.8111 s


In [10]:
len(amounts_converted)

10000000

## Part 3: Converting Currencies in Parallel

In [11]:
import numpy as np
import joblib
joblib.cpu_count()

2

In [12]:
parallel_runner = joblib.Parallel(n_jobs=2, batch_size=10000)
par_start = time.time()
amounts_cleaned_parallel = parallel_runner(
  joblib.delayed(converter.convert)(a) for a in amounts
)
par_end = time.time()
disp_time(par_start, par_end)

KeyboardInterrupt: 